# Import

In [2]:
!pip install azure-identity

Looking in indexes: https://devpi.com.stihl-dns.net/mbtech/dev/+simple/
     ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
     ---------------- ----------------------- 1.3/3.2 MB 7.4 MB/s eta 0:00:01
     ----------------------------------- ---- 2.9/3.2 MB 7.0 MB/s eta 0:00:01
     ---------------------------------------- 3.2/3.2 MB 7.0 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import sys
import os
import requests
import json
from azure.identity import InteractiveBrowserCredential

# if you face any issues with the import of modules, make sure, that the path up to '...\UseCaseTemplate\src' is part of the sys.path's
# if this is not the case, just add the path with the following line
# therefore, you need to replace <REPLACE_WITH_YOUR_PATH> with your path to the folder \UseCaseTemplate\src
# sys.path.append(r"c:\<REPLACE_WITH_YOUR_PATH>\UseCaseTemplate\src")
# for p in sys.path:
#     print(p)

from application.utils.io import write_json
from application.utils.auth import get_keyvault_secrets

# Variables

In [7]:
# Variable for environment
env = 'dt'

# Variable for environment of run execution
run_environment = 'vs_code'
if run_environment not in ['vs_code', 'synapse']:
    raise ValueError(f"Invalid value for run_environment: {run_environment}. Must be 'vs_code' or 'synapse'.")

# Variable for KeyVault Name which contains the secret for the Storage Account
keyvault_name = f"kvaultalbsig{env}"

# CONFIGURATION FILES

In [8]:
# configuration files contain information about source and destination of data
base_dir = os.getcwd()  # path of current ipynb file
config_path = os.path.join(base_dir, "config", "dt")

config_path_dstn = os.path.join(config_path, "dstn_adls.json")
config_path_src = os.path.join(config_path, "src.json")

with open(config_path_dstn, "r") as f:
    dstn_config = json.load(f)

with open(config_path_src, "r") as f:
    src_config = json.load(f)

# AUTHENTICATION

In [10]:
credential = InteractiveBrowserCredential(additionally_allowed_tenants="*")

# Log in with your Azure Account to receive stored secret in KeyVault 
dstn_storage_account_secret = get_keyvault_secrets(
    keyvault_name = keyvault_name,                                      # kvaultalbsigdt
    run_environment = run_environment,
    secret_name = dstn_config.get('storage_account_secret_name'),       # 01bronzeadlsdt-ak
    credential = credential,
)

# print(dstn_storage_account_secret)

# READ DATA

In [11]:
# Yahoo Finance API Base URL - "https://query1.finance.yahoo.com/v8/finance/chart/"
api_base_url = src_config.get('api_base_url')

ticker_lst = [
    'AAPL',             # Apple
    'NVDA',             # Nvidia
    'MSFT',             # Microsoft
    'META',             # Meta
    'AMZN',             # Amazon
    'SXR8.DE'           # S&P 500 ETF
]
for ticker in ticker_lst:
    print(f'Processing {ticker} data.')
    current_api_url = api_base_url + f'{ticker}'
    params = {
        'range': src_config.get('period'),
        'interval': src_config.get('interval'),
    }
    # common user agent string that simulates a modern web browser
    headers = {
        'User-Agent': 'Mozilla/5.0'
    }

# ---------------------------- TRANSFORM DATA ----------------------------
    response = requests.get(current_api_url, params=params, headers=headers)
    data = response.json()
    print(data)

    if 'chart' not in data or 'result' not in data['chart']:
        raise Exception(f"Failed to fetch data for {ticker}. Response: {data}")

# ---------------------------- WRITE DATA ----------------------------
    current_file_path = dstn_config.get('file_path') + f'{ticker}_raw.json'
    # write_json(
    #     data = data,
    #     storage_account_name = dstn_config.get('storage_account_name'),     # 01bronzeadlsdt
    #     storage_account_secret = dstn_storage_account_secret,               # 01bronzeadlsdt-ak
    #     container_name = dstn_config.get('container_name'),                  
    #     file_path = current_file_path,                                      
    #     )

    print(
        f"JSON file for {ticker} successfully written to {dstn_config.get('container_name')}/{current_file_path} in {dstn_config.get('storage_account_name')}."
    )

Processing AAPL data.
{'chart': {'result': [{'meta': {'currency': 'USD', 'symbol': 'AAPL', 'exchangeName': 'NMS', 'fullExchangeName': 'NasdaqGS', 'instrumentType': 'EQUITY', 'firstTradeDate': 345479400, 'regularMarketTime': 1746635502, 'hasPrePostMarketData': True, 'gmtoffset': -14400, 'timezone': 'EDT', 'exchangeTimezoneName': 'America/New_York', 'regularMarketPrice': 194.754, 'fiftyTwoWeekHigh': 260.1, 'fiftyTwoWeekLow': 169.21, 'regularMarketDayHigh': 199.44, 'regularMarketDayLow': 193.25, 'regularMarketVolume': 29140968, 'longName': 'Apple Inc.', 'shortName': 'Apple Inc.', 'chartPreviousClose': 181.46, 'priceHint': 2, 'currentTradingPeriod': {'pre': {'timezone': 'EDT', 'end': 1746624600, 'start': 1746604800, 'gmtoffset': -14400}, 'regular': {'timezone': 'EDT', 'end': 1746648000, 'start': 1746624600, 'gmtoffset': -14400}, 'post': {'timezone': 'EDT', 'end': 1746662400, 'start': 1746648000, 'gmtoffset': -14400}}, 'dataGranularity': '1d', 'range': '1mo', 'validRanges': ['1d', '5d', '1m